# TP3 — RAG multimodal agentique

Ce notebook suit les 7 étapes demandées avec le PDF **Attention Is All You Need**.
Les sorties générées sont placées dans `resultats_pdf/` (dossier ignoré par Git).

## Étape 1 — Extraction du texte et des images

In [1]:
from pathlib import Path
import hashlib
import json
import pymupdf

PDF_PATH = Path("pdfs") / "attention is all you need.pdf"
OUTPUT_DIR = Path("resultats_pdf")
TEXT_DIR = OUTPUT_DIR / "textes"
IMAGE_DIR = OUTPUT_DIR / "images"
OUTPUT_DIR.mkdir(exist_ok=True)
TEXT_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(exist_ok=True)

assert PDF_PATH.exists(), f"PDF introuvable : {PDF_PATH}"
document_hash = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()

pdf = pymupdf.open(PDF_PATH)
pages = []
image_records = []
for page_number, page in enumerate(pdf, start=1):
    text = page.get_text("text").strip()
    text_path = TEXT_DIR / f"page_{page_number:03d}.txt"
    text_path.write_text(text, encoding="utf-8")
    pages.append({"page": page_number, "text": text, "path": str(text_path)})

    for image_number, image in enumerate(page.get_images(full=True), start=1):
        image_data = pdf.extract_image(image[0])
        image_path = IMAGE_DIR / f"page_{page_number:03d}_image_{image_number:02d}.{image_data['ext']}"
        image_path.write_bytes(image_data["image"])
        image_records.append({"page": page_number, "path": str(image_path)})

extraction_manifest = {
    "document": str(PDF_PATH),
    "sha256": document_hash,
    "pages": len(pages),
    "images": len(image_records),
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(extraction_manifest, indent=2), encoding="utf-8")
print(f"Étape 1 terminée : {len(pages)} pages et {len(image_records)} images extraites.")

Étape 1 terminée : 11 pages et 3 images extraites.
